# **Cloning Repo and Importing Libraries**

In [1]:
!git clone https://github.com/Neural-GPT/Medical-Rag-Chatbot.git

Cloning into 'Medical-Rag-Chatbot'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 100 (delta 44), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 6.06 MiB | 11.11 MiB/s, done.
Resolving deltas: 100% (44/44), done.


In [2]:
import os
os.chdir("/content/Medical-Rag-Chatbot")

In [3]:
!pip install -r requirements.txt # you might have to restart session, no need to rerun this cell

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [4]:
import os
import sys
import requests
import pandas as pd
from tqdm import tqdm
from groq import Groq

from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.document_loaders import DataFrameLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
from src.preprocessing import load_data, split_docs
from src.vector import retrieve, generate_vector_store
from src.ask_rag import ask_rag

# **Preprocessing and Vector Database Generation**

In [7]:
data_path = "data/pubmed_diabetes_1000_meta.csv"

documents = load_data(data_path, DataFrameLoader)
text = split_docs(documents, RecursiveCharacterTextSplitter, chunk_size = 850, chunk_overlap = 150)

Total document chunks created: 2852


In [9]:
generator = HuggingFaceEmbeddings(model_name = "BAAI/bge-large-en-v1.5", model_kwargs = {"device": "cuda"})
vector_db_path = "src/Vector_db_850_large" # you can use pre built vector db (850 chunk size and 150 overlap) to save time

vector_store = generate_vector_store(generator, vector_db_path, text, FAISS)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# **Retriever and LLM Inference**

In [10]:
query = "What are the causes of diabetes?"
retriever = vector_store.as_retriever(search_type = "mmr", k = 4, fetch_k = 8, lambda_mult = 0.75) # lambda = [0,1]: 1 for pure relevance, 0 for diversity

context = retrieve(query, retriever)

for c in context:
  print(f"{c}\n")

Background: Diabetes is referred to a group of diseases characterized by high glucose levels in blood. It is caused by a deficiency in the production or function of insulin or both, which can occur because of different reasons, resulting in protein and lipid metabolic disorders. The aim of this study was to systematically review the prevalence and incidence of type 1 diabetes in the world.

The common causes of central diabetes insipidus (CDI) include trauma to the pituitary, hypoperfusion, and malignancy. However, CDI can also be transient. An emerging cause of transient diabetes insipidus is through the use and withdrawal of vasopressin. Here, we present a case of transient CDI that developed during an intensive care unit admission.

diabetes mellitus (e.g., metformin and thiazolidinediones) has been broadly evaluated in cancer prevention. Thus, a better understanding of the links between type 2 diabetes mellitus and cancer will help to identify the contributing factors and the patho

In [11]:
with open("/content/GROQ_API_KEY.txt") as f: # use your own API key (txt format)
    os.environ["GROQ_API_KEY"] = f.read().strip()

client = Groq(api_key = os.environ["GROQ_API_KEY"])

model = "qwen/qwen3-32b"

template = """
    "You are a medical Q&A assistant. Answer the question using ONLY the provided context."
    "Be concise and accurate. If the context doesn't contain the answer, say I don't know."
    "Do not add information beyond what is in the context."

    Context: {context}
    Question: {question}
    Answer:
      """


query = "What is diabetes?"

In [12]:
ask_rag(query, retriever, template, client, model)

'<think>\nOkay, let\'s tackle this question. The user is asking "What is diabetes?" and I need to answer based solely on the provided context. Let me go through the context carefully.\n\nFirst, the context starts by stating that diabetes mellitus is a chronic heterogeneous metabolic disorder with complex pathogenesis. It\'s characterized by elevated blood glucose (hyperglycemia) due to issues with insulin secretion, insulin action, or both. The hyperglycemia leads to problems with carbohydrate, fat, and protein metabolism. Long-term issues include microvascular and macrovascular complications. It also mentions that hyperglycemia is the primary biomarker for diabetes diagnosis.\n\nOther parts of the context talk about the global impact, but the question is just about defining diabetes. The key points here are the definition as a chronic metabolic disorder, elevated blood glucose, and the causes related to insulin. The answer should include that it\'s characterized by hyperglycemia from 

In [ ]:
"""
Diabetes mellitus is a chronic heterogeneous metabolic disorder characterized by elevated blood glucose levels (hyperglycemia), resulting from abnormalities
in insulin secretion, insulin action, or both. It leads to metabolic dysfunctions involving carbohydrates, fats, and proteins, and hyperglycemia
is the primary biomarker for its diagnosis.

"""

# **Evaluation [LLM as Judge: ChatGPT]**

In [13]:
retrieve("What are the causes of diabetes?", retriever)

['Background: Diabetes is referred to a group of diseases characterized by high glucose levels in blood. It is caused by a deficiency in the production or function of insulin or both, which can occur because of different reasons, resulting in protein and lipid metabolic disorders. The aim of this study was to systematically review the prevalence and incidence of type 1 diabetes in the world.',
 'The common causes of central diabetes insipidus (CDI) include trauma to the pituitary, hypoperfusion, and malignancy. However, CDI can also be transient. An emerging cause of transient diabetes insipidus is through the use and withdrawal of vasopressin. Here, we present a case of transient CDI that developed during an intensive care unit admission.',
 'diabetes mellitus (e.g., metformin and thiazolidinediones) has been broadly evaluated in cancer prevention. Thus, a better understanding of the links between type 2 diabetes mellitus and cancer will help to identify the contributing factors and t

In [14]:
ask_rag("What are the causes of diabetes?", retriever, template, client, model)

'<think>\nOkay, the user is asking about the causes of diabetes. Let me check the provided context to make sure I only use that information. \n\nFirst, the context mentions diabetes as a group of diseases with high blood glucose due to insulin production or function issues. It mentions different causes leading to metabolic disorders. So that\'s the general background.\n\nThen there\'s a section on central diabetes insipidus (CDI), noting causes like pituitary trauma, hypoperfusion, malignancy, and transient cases from vasopressin use. Wait, but CDI is different from diabetes mellitus, right? The user is asking about diabetes in general, which is typically diabetes mellitus. The context also brings up type 1 diabetes prevalence, but the question is about causes, so maybe I should focus on the main diabetes types.\n\nAnother part talks about type 2 diabetes mellitus (T2DM) and its links to cancer, mentioning drugs like metformin. However, the question is about causes, not treatments or r

In [16]:
"""
Response: (i'll remove the think part before evaluation)

Answer:
The causes of diabetes include:
1. **Diabetes mellitus**: Deficiency in insulin production or function (leading to glucose, protein, and lipid metabolic disorders), as seen in type 1 and type 2 diabetes.
2. **Neonatal diabetes**: Genetic mutations affecting pancreatic beta cell function.
3. **Central diabetes insipidus (CDI)**: Trauma to the pituitary, hypoperfusion, malignancy, or transient cases linked to vasopressin use/withdrawal.

Note: The context distinguishes between diabetes mellitus (glucose disorders) and diabetes insipidus (fluid balance disorders).

"""

"\nResponse: (i'll remove the think part before evaluation)\n\nAnswer:  \nThe causes of diabetes include:  \n1. **Diabetes mellitus**: Deficiency in insulin production or function (leading to glucose, protein, and lipid metabolic disorders), as seen in type 1 and type 2 diabetes.  \n2. **Neonatal diabetes**: Genetic mutations affecting pancreatic beta cell function.  \n3. **Central diabetes insipidus (CDI)**: Trauma to the pituitary, hypoperfusion, malignancy, or transient cases linked to vasopressin use/withdrawal.  \n\nNote: The context distinguishes between diabetes mellitus (glucose disorders) and diabetes insipidus (fluid balance disorders).\n\n"

In [ ]:
query = "What are the causes of diabetes?"

context = ['Background: Diabetes is referred to a group of diseases characterized by high glucose levels in blood. It is caused by a deficiency in the production or function of insulin or both, which can occur because of different reasons, resulting in protein and lipid metabolic disorders. The aim of this study was to systematically review the prevalence and incidence of type 1 diabetes in the world.',
 'The common causes of central diabetes insipidus (CDI) include trauma to the pituitary, hypoperfusion, and malignancy. However, CDI can also be transient. An emerging cause of transient diabetes insipidus is through the use and withdrawal of vasopressin. Here, we present a case of transient CDI that developed during an intensive care unit admission.',
 'diabetes mellitus (e.g., metformin and thiazolidinediones) has been broadly evaluated in cancer prevention. Thus, a better understanding of the links between type 2 diabetes mellitus and cancer will help to identify the contributing factors and the pathophysiological pathways and to design personalized preventive strategies. The final goal is to facilitate healthy aging and the prevention of cancer and other diseases related with type 2 diabetes mellitus, which are among the main sources of disability and death in the European Union and worldwide.',
 'Neonatal diabetes is a rare cause of hyperglycemia in the neonatal period. It is caused by mutations in genes that encode proteins playing critical roles in normal functions of pancreatic beta cells. Neonatal diabetes is divided into temporary and permanent subtypes. Treatment is based on the correction of fluid-electrolyte disturbances and hyperglycemia. Patients respond to insulin or sulfonylurea treatment according to the mutation type. Close glucose monitoring and education of caregivers about diabetes are vital.']

response = "Answer:  "
"The causes of diabetes include:  "
"1. **Diabetes mellitus**: Deficiency in insulin production or function (leading to glucose, protein, and lipid metabolic disorders), as seen in type 1 and type 2 diabetes."
"2. **Neonatal diabetes**: Genetic mutations affecting pancreatic beta cell function.  "
"3. **Central diabetes insipidus (CDI)**: Trauma to the pituitary, hypoperfusion, malignancy, or transient cases linked to vasopressin use/withdrawal.  "
"Note: The context distinguishes between diabetes mellitus (glucose disorders) and diabetes insipidus (fluid balance disorders)."

In [ ]:
# now ill pass the above data to chatgpt and ask it to judge the rag system and rate using suitable metrics

| Metric                    | Score (0–10) | Notes                                                                                                                                                        |
| ------------------------- | ------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **Relevance**             | 10           | Every part of the answer is supported by the context.                                                                                                        |
| **Completeness**          | 9            | Covers diabetes mellitus, neonatal diabetes, and CDI; could better separate type 1 vs type 2 causes.                                                         |
| **Accuracy**              | 9            | Correct per context; minor phrasing could be sharper (“impaired insulin secretion and/or action” instead of “deficiency in insulin production or function”). |
| **Clarity / Conciseness** | 9            | Structured and readable; “Note” could be integrated more smoothly.                                                                                           |
